# docagent on Google Colab (vLLM)

Run top to bottom on a GPU runtime (Runtime → Change runtime type → GPU). Expects `senior-data-scientist.zip` and, optionally, `store-backup.tar.gz` in `MyDrive/ColabProjects/`.

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

## 2. Unpack the project (make sure Project zip is in MyDrive/ColabProjects)

In [ ]:
!rm -rf /content/senior-data-scientist
!unzip -oq /content/drive/MyDrive/ColabProjects/senior-data-scientist.zip -d /content
%cd /content/senior-data-scientist

## 3. Restore `store/` from the Drive backup (skipped on the first run)

In [ ]:
import shutil
from pathlib import Path
import subprocess

drive_backup = Path(
    "/content/drive/MyDrive/ColabProjects/store-backup.tar.gz"
)
local_archive = Path("/content/store-backup.tar.gz")
project = Path("/content/senior-data-scientist")

project.mkdir(parents=True, exist_ok=True)

if drive_backup.is_file():
    shutil.copy2(drive_backup, local_archive)
    subprocess.run(
        ["tar", "-xzf", str(local_archive), "-C", str(project)],
        check=True
    )
    print("Store restored successfully.")
else:
    # First run: no backup yet. Cell 9 builds the index from scratch, cell 10 saves the backup.
    print(f"No backup at {drive_backup} - starting with an empty store (first run).")

## 4. Install vLLM and the docagent package

In [ ]:
%cd /content/senior-data-scientist

!pip install -q vllm==0.31.0 && pip install -q -e .

## 5. Check the GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,memory.used --format=csv

## 6. Pick the config and start the matching vLLM server

In [ ]:
import os, yaml
%cd /content/senior-data-scientist

# Pick the agent version here; the vLLM server is started to match it.
CONFIG = "configs/colab_vllm_awq.yaml"   # or "configs/colab_vllm.yaml"
# CONFIG = "configs/colab_vllm.yaml"
MAX_LEN = 8192                           # vLLM --max-model-len (16384 if prompts get rejected for length)
GPU_UTIL = 0.90                          # share of GPU memory for vLLM; the rest is for bge-m3 on cuda

os.environ["DOCAGENT_CONFIG"] = CONFIG   # CLI, chat app and Agent() all read this
SERVED = yaml.safe_load(open(CONFIG))["llm"]["model"]   # vLLM must serve exactly the name the config asks for

print(SERVED)
if SERVED.endswith("-awq"):
    # AWQ 4-bit weights in fp16: fits a T4
    print(f"{CONFIG}: serving AWQ as {SERVED}")
    !MAX_LEN={MAX_LEN} GPU_UTIL={GPU_UTIL} MODEL=Qwen/Qwen2.5-VL-7B-Instruct-AWQ QUANT=awq DTYPE=half SERVED={SERVED} bash scripts/start_vllm.sh
else:
    # start_vllm.sh picks the weights for the GPU: bf16 on L4/A100, AWQ on a T4
    print(f"{CONFIG}: serving as {SERVED}, weights chosen by GPU")
    !MAX_LEN={MAX_LEN} GPU_UTIL={GPU_UTIL} SERVED={SERVED} bash scripts/start_vllm.sh

## 7. Check the server responds

In [ ]:
!curl -s http://127.0.0.1:8000/v1/models | head -c 200

## 8. Download the embedding model (bge-m3)

In [ ]:
!python scripts/download_models.py

## 9. Ingest the reports

In [ ]:
!python -m docagent.cli ingest data/reports
!python -m docagent.cli status

## 10. Launch the chat app (public Gradio share link)

In [ ]:
import sys
sys.path.insert(0, "/content/senior-data-scientist/src")
sys.path.insert(0, "/content/senior-data-scientist/app")

import chat
from IPython.display import HTML, display

demo = chat.demo

demo.launch(
    server_name="127.0.0.1",
    server_port=7860,
    share=True,
    prevent_thread_lock=True,
    inline=False,
)

url = demo.share_url
display(HTML(f'<a href="{url}" target="_blank">{url}</a>'))

## 11. Copy `store/` to Drive as a folder

In [ ]:
import shutil
from pathlib import Path

project = Path("/content/senior-data-scientist")
archive = Path("/content/store-backup.tar.gz")
drive_backup = Path(
    "/content/drive/MyDrive/ColabProjects/store-backup.tar.gz"
)

# Create one compressed archive of the entire store folder.
!tar -czf /content/store-backup.tar.gz \
    -C /content/senior-data-scientist store

# Copy the archive to persistent storage.
shutil.copy2(archive, drive_backup)

print(f"Backup saved: {drive_backup}")
print(f"Archive size: {archive.stat().st_size / (1024**3):.2f} GB")

## 12. Save run outputs, eval results, traces and logs to Drive

In [ ]:
!mkdir -p /content/drive/MyDrive/ColabProjects/run_out
!cp -r outputs eval/results store/traces.db store/logs vllm.log /content/drive/MyDrive/ColabProjects/run_out/

## 13. Trace of the latest request

In [ ]:
!python -m docagent.cli trace last

## 14. Spans of one request (replace the request id)

In [ ]:
import sqlite3, pandas as pd
con = sqlite3.connect("store/traces.db")
pd.read_sql("select * from spans where request_id = '53b8c2c82def' order by started_at", con)

## 15. Ask a few questions in the notebook

In [ ]:
import sys, time
sys.path.insert(0, "/content/senior-data-scientist/src")
from IPython.display import Markdown, display
from docagent.agent import Agent
from docagent.render import render_markdown

questions = [
    "What is the fatigue life of a 2024-T3 spar in cycles?",
    "Why did the left wing main spar fail?",
    "What caused the throttle arm to fracture?",
]
agent = chat.agent if "chat" in globals() else Agent()
batch_id = "questions-" + time.strftime("%Y%m%d-%H%M%S")
for q in questions:
    resp = agent.ask(q, batch_id=batch_id)
    display(Markdown(f"### {q}\n\n" + render_markdown(resp)))
print("traces:", batch_id)